In [ ]:
import polars as pl
import httpx
from dotenv import load_dotenv
import os

load_dotenv()

PURPLE_AIR_API_KEY = os.getenv("PURPLE_AIR_API_KEY")

In [ ]:
# get json data from purple air response - note this code is from httpx documentation and just modified to work with this project's usecase

try:
    sensors = httpx.get(
        "https://api.purpleair.com/v1/sensors",
        params={
            "api_key": PURPLE_AIR_API_KEY,
            "fields": "sensor_index,latitude,longitude,position_rating,location_type,last_seen,pm2.5_atm,temperature,humidity",
            "max_age": 0,
            "nwlat": 40.962891,
            "nwlng": -74.264395,
            "selat": 40.470116,
            "selng": -73.675450,
        },
    )
    if sensors.status_code == 200:
        sensors_json = sensors.json()
        print(sensors_json)
    else:
        print(f"Request failed with status code: {sensors.status_code}, response: {sensors.text}")
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to make the request: {e}")

{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1774753174, 'data_time_stamp': 1774753143, 'max_age': 0, 'firmware_default_version': '7.02', 'fields': ['sensor_index', 'last_seen', 'location_type', 'position_rating', 'latitude', 'longitude', 'humidity', 'temperature', 'pm2.5_atm'], 'location_types': ['outside', 'inside'], 'data': [[262999, 1774753104, 1, 0, 40.801434, -73.970535, 19, 83, 0.8], [263287, 1755525360, 1, 5, 40.842293, -74.07955, 51, 71, 1.2], [263301, 1755525237, 1, 5, 40.842327, -74.079544, 51, 72, 2.6], [263457, 1755525786, 1, 5, 40.84218, -74.07933, None, None, 2.7], [1660, 1663947634, 0, 5, 40.721, -74.1928, 36, 65, 0.0], [2319, 1666785882, 0, 5, 40.809963, -73.95416, 76, 73, 10.4], [264499, 1762224167, 1, 5, 40.716164, -73.96324, 32, 77, 6.4], [2897, 1518323822, 0, 5, 40.790447, -73.733284, 86, 50, 128.0], [265475, 1774147551, 1, 0, 40.682373, -73.97324, 28, 84, 5.6], [265803, 1773728621, 0, 0, 40.686188, -73.94205, None, None, 0.1], [4803, 1774752686, 0, 5, 40.787937,

In [3]:
raw_sensors_df = pl.DataFrame(data = sensors_json["data"], \
                  schema = sensors_json["fields"])

print(raw_sensors_df.height)
raw_sensors_df.head()

595


C:\Users\crist\AppData\Local\Temp\ipykernel_36728\315921795.py:1: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  raw_sensors_df = pl.DataFrame(data = sensors_json["data"], \


sensor_index,last_seen,location_type,position_rating,latitude,longitude,humidity,temperature,pm2.5_atm
i64,i64,i64,i64,f64,f64,i64,i64,f64
262999,1774753104,1,0,40.801434,-73.970535,19,83,0.8
263287,1755525360,1,5,40.842293,-74.07955,51,71,1.2
263301,1755525237,1,5,40.842327,-74.079544,51,72,2.6
263457,1755525786,1,5,40.84218,-74.07933,null,null,2.7
1660,1663947634,0,5,40.721,-74.1928,36,65,0.0


### Cleaning data

Since we are looking to create a dashboard that shows air quality in NYC, we need to firstly clean the data and do a spatial join to get sensors only within the 5 boroughs,
and also remove the sensors which are in indoors (location_type = 1) and those with a position rating of 0 (position_rating = 0) since they are likely to be inaccurate. We will also convert the last_seen column to a datetime format.

In [ ]:
import geopandas as gpd

raw_sensors_gdf = gpd.GeoDataFrame(
                    raw_sensors_df.to_pandas(),
                    geometry=gpd.points_from_xy(raw_sensors_df["longitude"], raw_sensors_df["latitude"]),
                    crs="EPSG:4326"
                )

raw_sensors_gdf = raw_sensors_gdf.loc[(raw_sensors_gdf["position_rating"] > 0) & (raw_sensors_gdf["location_type"] != 1)]

raw_sensors_gdf.head()

,sensor_index,last_seen,location_type,position_rating,latitude,longitude,humidity,temperature,pm2.5_atm,geometry
4,1660,1663947634,0,5,40.721000,-74.192800,36.0,65.0,0.0,POINT (-74.1928 40.721)
5,2319,1666785882,0,5,40.809963,-73.954160,76.0,73.0,10.4,POINT (-73.95416 40.80996)
7,2897,1518323822,0,5,40.790447,-73.733284,86.0,50.0,128.0,POINT (-73.73328 40.79045)
10,4803,1774752686,0,5,40.787937,-73.968070,NaN,NaN,2.9,POINT (-73.96807 40.78794)
15,5648,1555619294,0,5,40.735424,-74.009460,100.0,NaN,19.6,POINT (-74.00946 40.73542)


In [5]:
# Now that the data is in a geodataframe, we can do a spatial join to remove sensors outside of the 5 boroughs of NYC.
# The borough boundaries data is in the NYC Open Data portal.

response = httpx.get("https://data.cityofnewyork.us/resource/gthc-hcne.geojson")
response.raise_for_status()

geojson = response.json()

boroughs_gdf = gpd.GeoDataFrame.from_features(geojson["features"], crs="EPSG:4326")
boroughs_gdf.head()


,geometry,borocode,boroname,shape_area,shape_leng
0,"MULTIPOLYGON (((-74.05051 40.56642, -74.05047 ...",5,Staten Island,1623618358.46,325912.288988
1,"MULTIPOLYGON (((-74.01093 40.68449, -74.01193 ...",1,Manhattan,636631650.451,359537.866079
2,"MULTIPOLYGON (((-73.89681 40.79581, -73.89694 ...",2,Bronx,1187199300.36,463147.071867
3,"MULTIPOLYGON (((-73.86327 40.58388, -73.86381 ...",3,Brooklyn,1934462607.43,726953.044632
4,"MULTIPOLYGON (((-73.82645 40.59053, -73.82642 ...",4,Queens,3041419178.99,887905.076018


In [ ]:
# Get Sensors Data API End Point - to be used in Kestra Flow
# Store a wide-but-reasonable schema in the lake so dbt can build 'typical pattern' aggregates (time-of-day/season/borough).
#
# Column intent (high-level):
# - IDs + geo: keys + spatial join to NYC borough boundaries + clustering in BigQuery
# - Status/filtering: exclude indoor sensors + low-quality location (position_rating) + private sensors if desired
# - Timestamps: last_seen drives event-time; last_modified/date_created help debugging changes/new sensors
# - QA flags + confidence: exclude downgraded channels / suspicious readings when building 'typical' averages
# - PM + env metrics: the actual dashboard measures + covariates for seasonality/time-of-day patterns
# - Device health: troubleshoot sensors with latency/weak WiFi (helps explain weird values or dropouts)
#
# Note on naming: PurpleAir includes dots in some field names (e.g. pm2.5_atm).
# When landing to BigQuery, you’ll typically normalize column names (e.g. replace '.' with '_') in ingestion or dbt.

try:
    sensors = httpx.get(
        "https://api.purpleair.com/v1/sensors",
        params={
            "api_key": PURPLE_AIR_API_KEY,
            "fields": (
                # IDs + geo
                "sensor_index,name,latitude,longitude,altitude,"
                # Status / filtering
                "location_type,private,position_rating,"
                # Timestamps / provenance
                "last_seen,last_modified,date_created,"
                # QA: channel downgrade flags + confidence
                "channel_state,channel_flags,channel_flags_manual,channel_flags_auto,"
                "confidence,confidence_manual,confidence_auto,"
                # Core air quality metrics (mass concentration)
                "pm1.0_atm,pm2.5_atm,pm2.5_atm_a,pm2.5_atm_b,pm2.5_alt,pm10.0_atm,"
                # Environmental context
                "humidity,temperature,pressure,"
                # Device health / troubleshooting
                "rssi,uptime,pa_latency"
            ),
            "max_age": 0,
            "nwlat": 40.962891,
            "nwlng": -74.264395,
            "selat": 40.470116,
            "selng": -73.675450,
        },
        timeout=30.0,
    )
    if sensors.status_code == 200:
        sensors_json = sensors.json()
        print(sensors_json)
    else:
        print(f"Request failed with status code: {sensors.status_code}, response: {sensors.text}")
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to make the request: {e}")